# Data pipeline check

Exercises the real `weather_fno` package code (not a rewrite of it) to
confirm the data pipeline is doing the right thing before spending compute
on an actual training or inference run:

1. Build the coarse train/val datasets (`data/split.py`) — flips latitude,
   normalises, caches to disk.
2. Build inference data for both configured targets (`inference/predict.py`)
   — one derives relative humidity from specific humidity + temperature,
   the other reads it directly but flips latitude like the coarse data.
3. Plot the same variable (2m temperature) from all three side by side, to
   visually confirm orientation is correct.
4. Plot the derived relative-humidity field on its own, to confirm the
   derivation produces something physically plausible.

Needs network access to GCS (public, anonymous — see
`weather_fno/data/io.py`) and the project's dependencies installed
(`pip install -r requirements.txt && pip install -e .` from the project
root). The first run will pull real data and may take a while; the coarse
dataset gets cached to `outputs/stats/train_cache.npz` /
`val_cache.npz` so re-running this notebook after the first time is fast
for step 1 (step 2 is not cached — it's a small, single-timestep pull per
target, so that's fine).

If anything here fails or looks wrong before you even get to the plots,
`python scripts/inspect_store.py --config configs/baseline_fno.yaml` is a
cheaper, metadata-only diagnostic — run that first.


In [1]:
import sys
from pathlib import Path

# Make `weather_fno` importable whether this notebook is run from
# `notebooks/` (the usual case) or from the project root, without
# requiring `pip install -e .` first.
_project_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(_project_root / "src"))

import numpy as np
import matplotlib.pyplot as plt

from weather_fno.config import load_config
from weather_fno.data.split import build_train_val_datasets
from weather_fno.data.preprocessing import denormalise
from weather_fno.data.io import open_dataset
from weather_fno.inference.predict import load_inference_data

CONFIG_PATH = _project_root / "configs" / "baseline_fno.yaml"
cfg = load_config(CONFIG_PATH)

def channel_index(short_name: str) -> int:
    return next(i for i, c in enumerate(cfg.data.channels) if c.short_name == short_name)

print(f"run_name: {cfg.run_name}")
print(f"channels: {len(cfg.data.channels)}")
print(f"inference targets: {[t.name for t in cfg.inference.targets]}")


run_name: fno_baseline_64x32
channels: 20
inference targets: ['native_highres', '1p5deg']


## 0. Orientation pre-flight check

Cheap, metadata-only (reads each store's latitude coordinate, no array data
downloaded) — run this BEFORE building any datasets below. For
`origin="lower"` to plot north-up correctly later, a source needs
`flip_lat` set so that row 0 ends up at the south pole after flipping
(i.e. ascending). If a source's raw order is `ascending`, it needs
`flip_lat: false`; if `descending`, it needs `flip_lat: true`. If any
`MISMATCH` prints below, fix that target's `flip_lat` in
`configs/baseline_fno.yaml` and re-run this cell before going any further
— it will save you from building (and plotting) a full dataset only to
find it's upside down.


In [ ]:
sources = [("coarse/train", cfg.data.gcs_bucket_path, cfg.data.flip_lat)]
sources += [(t.name, t.gcs_bucket_path, t.flip_lat) for t in cfg.inference.targets]

for label, gcs_path, configured_flip in sources:
    ds = open_dataset(gcs_path)
    lat = ds[cfg.data.lat_dim].values
    raw_ascending = lat[0] < lat[-1]
    final_ascending = raw_ascending if not configured_flip else (not raw_ascending)
    status = "OK" if final_ascending else "MISMATCH -- will plot upside-down, fix flip_lat"
    print(f"{label:14s} raw lat: {lat[0]:7.2f} -> {lat[-1]:7.2f} "
          f"({'ascending' if raw_ascending else 'descending'}), "
          f"configured flip_lat={configured_flip}  ->  {status}")


## 1. Build the coarse training/validation datasets

This exercises `data/io.py` (GCS open), `data/gcs_dataset.py` (channel selection, flip, normalise), and `data/split.py` (time-based split + stats reuse) end to end.

In [2]:
train_ds, val_ds = build_train_val_datasets(cfg.data)

print(f"train_ds: {train_ds.data.shape}  (T, C, H, W)")
print(f"val_ds:   {val_ds.data.shape}")
print(f"lat_values: {train_ds.lat_values[0]:.2f} -> {train_ds.lat_values[-1]:.2f}  "
      f"({'ascending (south->north)' if train_ds.lat_values[0] < train_ds.lat_values[-1] else 'descending (north->south)'})")
print(f"configured flip_lat={cfg.data.flip_lat}, flip_lon={cfg.data.flip_lon}")


I0809 17:43:49.287987 47500400 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0809 17:43:49.350088 47500445 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)


train_ds: torch.Size([21916, 20, 32, 64])  (T, C, H, W)
val_ds:   torch.Size([2924, 20, 32, 64])
lat_values: 87.19 -> -87.19  (descending (north->south))
configured flip_lat=True, flip_lon=False


In [3]:
# --- sanity checks: shapes, NaNs, normalisation correctness, stats reuse ---

T, C, H, W = train_ds.data.shape
expected_W, expected_H = cfg.data.resolution  # resolution is [lon, lat] = [W, H]
assert C == len(cfg.data.channels) == cfg.model.in_channels, \
    f"channel count mismatch: data has {C}, config expects {len(cfg.data.channels)}/{cfg.model.in_channels}"
assert (H, W) == (expected_H, expected_W), \
    f"spatial shape mismatch: data is {(H, W)}, config resolution implies {(expected_H, expected_W)}"
print(f"[OK] shape (T={T}, C={C}, H={H}, W={W}) matches config (20 channels, {expected_H}x{expected_W})")

for name, ds in [("train", train_ds), ("val", val_ds)]:
    arr = ds.data.numpy()
    n_nan, n_inf = np.isnan(arr).sum(), np.isinf(arr).sum()
    assert n_nan == 0 and n_inf == 0, f"{name}: found {n_nan} NaNs / {n_inf} Infs"
    print(f"[OK] {name}: no NaNs/Infs across {arr.size:,} values")

# Per-channel mean should be ~0 and std ~1 on the TRAIN split specifically
# (val is normalised with train's stats, so it won't necessarily land at
# exactly mean 0 / std 1 — that's expected, not a bug).
train_arr = train_ds.data.numpy()
per_channel_mean = train_arr.mean(axis=(0, 2, 3))
per_channel_std = train_arr.std(axis=(0, 2, 3))
bad = np.where((np.abs(per_channel_mean) > 0.05) | (np.abs(per_channel_std - 1) > 0.05))[0]
if len(bad):
    print(f"[CHECK] channels with mean/std noticeably off from 0/1 after normalisation: "
          f"{[cfg.data.channels[i].short_name for i in bad]}")
else:
    print("[OK] every channel's train-split mean/std is ~0/~1 after normalisation")

# val must reuse train's stats, never fit its own
assert np.allclose(train_ds.stats["mean"], val_ds.stats["mean"]) and \
       np.allclose(train_ds.stats["std"], val_ds.stats["std"]), \
    "val_ds stats differ from train_ds stats — val should always reuse train's stats"
print("[OK] val split reuses train's normalisation stats (not re-fit)")

print("\nChannel index reference:")
for i, c in enumerate(cfg.data.channels):
    print(f"  {i:2d}  {c.short_name:6s} ({c.name}{'' if c.level is None else f' @ {c.level}hPa'})")


[OK] shape (T=21916, C=20, H=32, W=64) matches config (20 channels, 32x64)
[OK] train: no NaNs/Infs across 897,679,360 values
[OK] val: no NaNs/Infs across 119,767,040 values
[OK] every channel's train-split mean/std is ~0/~1 after normalisation
[OK] val split reuses train's normalisation stats (not re-fit)

Channel index reference:
   0  u10    (10m_u_component_of_wind)
   1  v10    (10m_v_component_of_wind)
   2  t2m    (2m_temperature)
   3  sp     (surface_pressure)
   4  mslp   (mean_sea_level_pressure)
   5  t850   (temperature @ 850hPa)
   6  u1000  (u_component_of_wind @ 1000hPa)
   7  v1000  (v_component_of_wind @ 1000hPa)
   8  z1000  (geopotential @ 1000hPa)
   9  u850   (u_component_of_wind @ 850hPa)
  10  v850   (v_component_of_wind @ 850hPa)
  11  z850   (geopotential @ 850hPa)
  12  u500   (u_component_of_wind @ 500hPa)
  13  v500   (v_component_of_wind @ 500hPa)
  14  z500   (geopotential @ 500hPa)
  15  t500   (temperature @ 500hPa)
  16  z50    (geopotential @ 50hPa)


## 2. Build inference data for both targets

Exercises `inference/preprocessing.py` (relative-humidity derivation) and `inference/predict.py::load_inference_data` for every target in `cfg.inference.targets` — each with its own flip settings and its own derive-vs-read-directly behaviour for relative humidity.

In [5]:
inference_arrays = {}
for target in cfg.inference.targets:
    print(f"loading target '{target.name}' ({target.gcs_bucket_path})...")
    arr = load_inference_data(cfg, target)
    inference_arrays[target.name] = arr
    print(f"  shape: {arr.shape}  (T, C, H, W) — flip_lat={target.flip_lat}, "
          f"flip_lon={target.flip_lon}, derive_relative_humidity={target.derive_relative_humidity}")


loading target 'native_highres' (gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721.zarr)...


KeyError: "No variable named 'TODO_specific_humidity_var'. Did you mean one of ('specific_humidity',)?"

In [ ]:
# --- sanity checks per target ---

r500_idx = channel_index("r500")
r850_idx = channel_index("r850")

for target in cfg.inference.targets:
    arr = inference_arrays[target.name]
    Tt, Ct, Ht, Wt = arr.shape
    assert Tt == 1, f"{target.name}: expected a single initial-condition timestep, got {Tt}"
    assert Ct == len(cfg.data.channels), f"{target.name}: channel count mismatch"

    expected_Wt, expected_Ht = target.resolution
    if (Ht, Wt) != (expected_Ht, expected_Wt):
        print(f"[CHECK] {target.name}: array shape {(Ht, Wt)} != configured resolution "
              f"{(expected_Ht, expected_Wt)} — confirm target.resolution in the config")
    else:
        print(f"[OK] {target.name}: spatial shape {(Ht, Wt)} matches configured resolution")

    n_nan, n_inf = np.isnan(arr).sum(), np.isinf(arr).sum()
    if n_nan or n_inf:
        print(f"[CHECK] {target.name}: found {n_nan} NaNs / {n_inf} Infs")
    else:
        print(f"[OK] {target.name}: no NaNs/Infs")

    for idx, label in [(r500_idx, "r500"), (r850_idx, "r850")]:
        rh = arr[0, idx]
        frac_clipped = ((rh <= 0.0) | (rh >= 100.0)).mean()
        print(f"  {label}: min={rh.min():.1f}  max={rh.max():.1f}  mean={rh.mean():.1f}  "
              f"clipped-at-bound={frac_clipped:.1%}"
              + ("  (derived from specific humidity)" if target.derive_relative_humidity else "  (read directly)"))
        if target.derive_relative_humidity and frac_clipped > 0.3:
            print(f"    [CHECK] >30% of {label} is clipped at 0 or 100 — likely a units mismatch "
                  f"feeding compute_relative_humidity (kg/kg vs g/kg, or K vs C)")


## 3. Side-by-side map comparison (2m temperature)

`t2m` is read directly (no derivation) at every resolution, so it's a clean
way to check axis orientation in isolation from the relative-humidity
derivation. Each panel uses `origin="lower"` with the array as our
pipeline actually produced it (after that source's own `flip_lat`/
`flip_lon`) — **a physically sane t2m field has warm colours in the middle
rows (tropics) and cool colours at the top/bottom rows (poles)**. If a
panel looks inverted (cool band down the middle, warm at top+bottom), that
target's `flip_lat` is wrong.


In [ ]:
def real_lat_lon(gcs_bucket_path: str):
    """Fetch this store's actual lat/lon coordinate BOUNDS, used only to
    label the plot axes with real degree values via `extent`. This does
    NOT determine north-up/south-up orientation -- that's controlled
    entirely by `origin="lower"` combined with the actual row order of the
    plotted array (i.e. by that source's flip_lat, already baked into the
    array by the pipeline). Reversing lat/lon here would be a no-op since
    only .min()/.max() are used below -- see the orientation pre-flight
    check earlier in this notebook for the actual flip_lat check.
    """
    ds = open_dataset(gcs_bucket_path)
    lat = ds[cfg.data.lat_dim].values
    lon = ds[cfg.data.lon_dim].values
    return lat, lon


t2m_idx = channel_index("t2m")

train_physical = denormalise(train_ds.data.numpy(), train_ds.stats)
train_t2m = train_physical[0, t2m_idx]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

panels = [("coarse (train)", train_t2m, cfg.data.flip_lat, cfg.data.flip_lon, cfg.data.gcs_bucket_path)]
for target in cfg.inference.targets:
    panels.append((target.name, inference_arrays[target.name][0, t2m_idx],
                    target.flip_lat, target.flip_lon, target.gcs_bucket_path))

for ax, (label, field, flip_lat, flip_lon, gcs_path) in zip(axes, panels):
    lat, lon = real_lat_lon(gcs_path)
    im = ax.imshow(field, origin="lower", cmap="viridis",
                    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
                    aspect="auto")
    ax.set_title(f"{label}\n{field.shape[1]}x{field.shape[0]}, flip_lat={flip_lat}")
    ax.set_xlabel("longitude")
    fig.colorbar(im, ax=ax, label="t2m (K)", fraction=0.046, pad=0.04)

axes[0].set_ylabel("latitude")
fig.suptitle("2m temperature — coarse training data vs. both inference targets")
fig.tight_layout()
plt.show()


## 4. Derived relative-humidity sanity check

Plots r500 for whichever target has `derive_relative_humidity=True` (the
native high-resolution store) — this is the field that's actually computed
by `compute_relative_humidity`, not read from the store, so it's worth
checking on its own. Expect broad moist bands near the equator (the ITCZ)
and dry subtropical bands/deserts (e.g. the Sahara, the Arabian peninsula)
— a field that's uniformly near 0%, uniformly near 100%, or visual noise
with no large-scale structure indicates a units problem in
`compute_relative_humidity`'s inputs.


In [ ]:
rh_target = next((t for t in cfg.inference.targets if t.derive_relative_humidity), None)

if rh_target is None:
    print("No target has derive_relative_humidity=True — nothing to check here.")
else:
    lat, lon = real_lat_lon(rh_target.gcs_bucket_path)
    rh_field = inference_arrays[rh_target.name][0, r500_idx]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    im = ax.imshow(rh_field, origin="lower", cmap="viridis", vmin=0, vmax=100,
                    extent=[lon.min(), lon.max(), lat.min(), lat.max()], aspect="auto")
    ax.set_title(f"derived r500 — {rh_target.name}")
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    fig.colorbar(im, ax=ax, label="relative humidity (%)")
    plt.show()


## Checklist — what "OK" vs. "a problem" looks like here

- **Shapes** (cell after step 1/2): should assert cleanly. If channel count
  is wrong, a channel spec's `name` doesn't match what's actually in a
  store (check with `inspect_store.py`).
- **NaN/Inf**: any non-zero count is a real problem — usually a bad
  `.sel(level=...)` (level not present on that variable) or a units issue
  feeding `compute_relative_humidity` (e.g. dividing near zero).
- **Train-split mean/std ~0/~1**: a channel far off this after
  normalisation usually means that channel's raw values are degenerate
  (constant, or dominated by a few outliers) — check it individually with
  `train_physical[:, idx]`.
- **t2m maps** (step 3): warm in the middle, cool at top+bottom, roughly
  symmetric about the equator. Upside-down = wrong `flip_lat` for that
  source. Mirrored left-right = wrong `flip_lon`. The three panels should
  look like recognisably the SAME field at different resolutions — same
  large-scale warm/cool pattern, just blockier at 64x32.
- **Derived r500** (step 4): large-scale moist/dry bands, not noise or a
  flat field. A `clipped-at-bound` fraction over ~30% (printed in the step-2
  sanity cell) is the strongest signal of a units bug — fix the
  `TODO_specific_humidity_var`/`TODO_temperature_var` variable names AND
  double-check their units (kg/kg vs g/kg, K vs °C) against what
  `inspect_store.py` reports for the native high-res store.


## Other things worth testing before running training/inference for real

Beyond what this notebook checks:

- **Actual specific-humidity/temperature variable names.** `predict.py`'s
  `load_inference_data` still has `"TODO_specific_humidity_var"` and
  `"TODO_temperature_var"` placeholders for the `native_highres` target —
  this notebook will raise a `KeyError` on step 2 until those are filled in
  with the real names from that store (`scripts/inspect_store.py`'s
  "Available data_vars" listing for that target will show them).
- **Units feeding `compute_relative_humidity`.** Confirm specific humidity
  is kg/kg (not g/kg) and temperature is Kelvin (not Celsius) in the native
  store — a silent factor-of-1000 or 273.15 error won't crash anything, it
  just produces a plausible-looking but wrong RH field. The
  clipped-at-bound fraction printed in step 2 is a decent first signal, but
  isn't proof either way.
- **Time alignment across sources.** The three panels in step 3 are each
  the FIRST available timestep in their own store/split — the coarse panel
  is `train_start` (2000-01-01), while each inference target starts from
  whatever its own store's first timestep is. These aren't necessarily the
  same calendar date. Fine for a pure orientation/sanity check, but not a
  like-for-like comparison of the same weather at the same time — if you
  want that, adapt step 2 to `.sel(time=...)` a shared date on all three
  before comparing.
- **`lat_dim`/`lon_dim` naming per store.** The notebook (and the whole
  pipeline) assumes all three stores use the same dimension names
  (`cfg.data.lat_dim`/`lon_dim`, currently `"latitude"`/`"longitude"`).
  `inspect_store.py` checks this explicitly per store — worth running once
  per new store you point the config at.
- **Longitude convention.** WeatherBench2 stores can use either `[0, 360)`
  or `[-180, 180)` — `inspect_store.py` reports which. If the two inference
  targets use different conventions from each other, comparing them side by
  side (as in step 3) can look subtly wrong even when both are internally
  correct. Not fixed automatically anywhere in the pipeline right now.
- **Level availability.** For any pressure-level channel (`t850`, `z500`,
  etc.), confirm the requested level actually exists on that variable in
  EVERY store you plan to use, not just the one you tested first —
  `inspect_store.py` flags `NOT IN [...]` per channel per store.
- **Memory/timing at full resolution once you go beyond a single
  timestep.** This notebook and the current `load_inference_data` only
  ever pull ONE timestep per target (see `CODE_REFERENCE.md`'s "Fixes
  applied" section for why). If you extend the pipeline later to compare
  multiple lead times against ground truth, re-check how much data that
  pulls from the native 1440x721 store before running it unattended.
